Step 2: Identify Product Type
Step 4: Extract Attributes Based on Product Template

In [ ]:
import openai
import pandas as pd
import json


# Set your API key securely
openai.api_key = "OPENAI_API_KEY"
 # Store key in environment variable

# Load template sheets without headers
pipe_template = pd.read_excel("ICE ENHANCEMENT PROJECT.xlsx", sheet_name="Pipe_Template", header=None)
flange_template = pd.read_excel("ICE ENHANCEMENT PROJECT.xlsx", sheet_name="Flange_template", header=None)

# Extract column names
pipe_columns = pipe_template.iloc[1].tolist()
flange_columns = flange_template.iloc[0].tolist()

def extract_attributes_with_llm(description, pipe_columns, flange_columns):
    prompt = f"""
You are an expert in extracting structured product attributes from technical product descriptions.

Given:
- A raw product description string.
- Two reference column lists:
   - `pipe_columns`: expected fields for pipe products.
   - `flange_columns`: expected fields for flange products.

Steps:
1. Identify the product type: PIPE or FLANGE.
2. Use the corresponding column list to determine which attributes to extract.
3. Parse the description and fill in values for each column.
4. If a value is not present, use "NA".
5. Return the result strictly as a JSON dictionary with column names as keys.

---
### 🔍 Reference Tables

#### `flange_df` Columns:
PRODUCT, NORM, CONSTRUCTION, SIZE1, SCHEDULE, GRADE, SIZE2, SCHEDULE2, PRESSURE_CLASS, LEVEL_CLASS, MATERIAL, ENDS, WALL_THICKNESS, WALL_THICKNESS2, OUTER_DIAMETER, OUTER_DIAMETER2, COATING, DIMEN_STAND

Sample Row:
FLANGE BLIND,	A182	,FORGED	,1	,STD,	F53,	8	,40/STD	,CLS 300,	CL1,	DS,	FF	,13.55	,3.91	,33.40,	219.10,	HDG to A153	,ASME B16.5

#### `pipe_df` Columns:
PRODUCT, NORM, CONSTRUCTION, SIZE1, SCHEDULE, GRADE, LEVEL_CLASS, MATERIAL, LENGTH, ENDS, WALL_THICKNESS, OUTER_DIAMETER, COATING, DIMEN_STAND


Sample Row:
PIPE,	A53/A106/API 5L,	SMLS,	6,40S	,B/X42	,PSL2	,CS,	DRL,	BE,	24.00	,168.30,	HDG to A123,	ASME B36.10M

### 🔡 Input

**Description**: "{description}"

**Pipe data**: {pipe_template}

**Flange data**: {flange_template}

---

🔁 Respond with **only a valid JSON object**.
"""
    try:
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            temperature=0.1,
            messages=[
                {"role": "system", "content": "You extract structured product attributes from descriptions."},
                {"role": "user", "content": prompt}
            ]
        )
        content = response["choices"][0]["message"]["content"]
        attributes_dict = json.loads(content)
        return attributes_dict
    except Exception as e:
        print(f"Error: {e}")
        print("Raw LLM Response:\n", content)
        return {}

# Sample product description
product_description = "3/4' BLIND FLANGE, CL1500, RF, ASTM A350-LF2 CL1, ASME B16.5, SOUR SERVICE"

# Run extraction
result = extract_attributes_with_llm(product_description, pipe_columns, flange_columns)
# Print product type if available
product_type = result.get("PRODUCT", "UNKNOWN").upper()
print(f" Detected Product Type: {product_type}")
print(json.dumps(result, indent=2))


 Detected Product Type: FLANGE
{
  "PRODUCT": "FLANGE",
  "NORM": "ASTM A350-LF2 CL1",
  "CONSTRUCTION": "NA",
  "SIZE1": "3/4",
  "SCHEDULE": "NA",
  "GRADE": "NA",
  "SIZE2": "NA",
  "SCHEDULE2": "NA",
  "PRESSURE_CLASS": "CL1500",
  "LEVEL_CLASS": "SOUR SERVICE",
  "MATERIAL": "NA",
  "ENDS": "RF",
  "WALL_THICKNESS": "NA",
  "WALL_THICKNESS2": "NA",
  "OUTER_DIAMETER": "NA",
  "OUTER_DIAMETER2": "NA",
  "COATING": "NA",
  "DIMEN_STAND": "ASME B16.5"
}


In [2]:
def validate_attributes(attributes_dict, pipe_template, flange_template, pipe_columns, flange_columns):
    product_type = attributes_dict.get("PRODUCT", "").strip().upper()

    # Choose correct template and columns
    if product_type == "PIPE":
        template_df = pipe_template
        template_columns = pipe_columns
        row_start_index = 2  # For pipe_template, skip first two rows
    elif product_type == "FLANGE":
        template_df = flange_template
        template_columns = flange_columns
        row_start_index = 1  # For flange_template, skip header row only
    else:
        print("Unknown product type; skipping validation.")
        return attributes_dict

    # Create a clean DataFrame from the valid rows
    template_data = template_df.iloc[row_start_index:].reset_index(drop=True)
    template_data.columns = template_columns

    validated_dict = {}

    for col in template_columns:
        value = attributes_dict.get(col, "NA")
        # Compare only if value is not already NA
        if value != "NA":
            if str(value).strip() not in template_data[col].astype(str).str.strip().unique():
                validated_dict[col] = "NA"
            else:
                validated_dict[col] = value
        else:
            validated_dict[col] = "NA"

    return validated_dict


check validation in sheet values is present in the sheet

In [61]:
# Run extraction
result = extract_attributes_with_llm(product_description, pipe_columns, flange_columns)

# Validate against the template
validated_result = validate_attributes(result, pipe_template, flange_template, pipe_columns, flange_columns)

# Print final result
print(f"\n✅ Final Validated Attributes:")
print(json.dumps(validated_result, indent=2))



✅ Final Validated Attributes:
{
  "PRODUCT": "FLANGE",
  "NORM": "NA",
  "CONSTRUCTION": "NA",
  "SIZE1": "3/4",
  "SCHEDULE": "NA",
  "GRADE": "NA",
  "SIZE2": "NA",
  "SCHEDULE2": "NA",
  "PRESSURE_CLASS": "NA",
  "LEVEL_CLASS": "NA",
  "MATERIAL": "NA",
  "ENDS": "RF",
  "WALL_THICKNESS": "NA",
  "WALL_THICKNESS2": "NA",
  "OUTER_DIAMETER": "NA",
  "OUTER_DIAMETER2": "NA",
  "COATING": "NA",
  "DIMEN_STAND": "ASME B16.5"
}


Step 5: Mandatory Parameter Validation

In [62]:
def validate_attributes_with_llm(attributes_dict):
    prompt = f"""
You are a product data validation expert.

Given the extracted attribute dictionary below, perform the following:
1. Identify if the product type is PIPE or FLANGE based on the "PRODUCT" field.
2. Validate that all mandatory fields for that product type are present and not "NA".
   - For PIPE: PRODUCT, NORM, GRADE, LEVEL_CLASS, SIZE1, SCHEDULE, CONSTRUCTION
   - For FLANGE: PRODUCT, NORM, GRADE, LEVEL_CLASS, SIZE1, PRESSURE_CLASS
   - Other fields such as SCHEDULE2, SIZE2 may also be mandatory in some special cases.
3. If any mandatory field is missing or has the value "NA", respond with:
   "Invalid Description - Missing: <list of missing fields>"
4. If all required fields are valid, respond with:
   "Valid Description"

Only return the validation message as plain text.
---
Attributes: {json.dumps(attributes_dict, indent=2)}
"""

    try:
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            temperature=0.1,
            messages=[
                {"role": "system", "content": "You validate product attributes based on mandatory field rules."},
                {"role": "user", "content": prompt}
            ]
        )
        validation_message = response["choices"][0]["message"]["content"].strip()
        return validation_message
    except Exception as e:
        print(f"LLM Validation Error: {e}")
        return "Validation Error - Unable to check attributes"


In [64]:


# Step 2: Validate against template sheet values (removes invalid template matches)
validated_result = validate_attributes(result, pipe_template, flange_template, pipe_columns, flange_columns)
print("\n✅ Template-Validated Attributes:")
print(json.dumps(validated_result, indent=2))

# Step 3: Validate business rules using LLM (based on validated result)
validation_status = validate_attributes_with_llm(validated_result)
print("\n📋 LLM Validation Result:")
print(validation_status)



✅ Template-Validated Attributes:
{
  "PRODUCT": "FLANGE",
  "NORM": "NA",
  "CONSTRUCTION": "NA",
  "SIZE1": "3/4",
  "SCHEDULE": "NA",
  "GRADE": "NA",
  "SIZE2": "NA",
  "SCHEDULE2": "NA",
  "PRESSURE_CLASS": "NA",
  "LEVEL_CLASS": "NA",
  "MATERIAL": "NA",
  "ENDS": "RF",
  "WALL_THICKNESS": "NA",
  "WALL_THICKNESS2": "NA",
  "OUTER_DIAMETER": "NA",
  "OUTER_DIAMETER2": "NA",
  "COATING": "NA",
  "DIMEN_STAND": "ASME B16.5"
}

📋 LLM Validation Result:
Invalid Description - Missing: NORM, GRADE, LEVEL_CLASS, PRESSURE_CLASS


In [65]:
# Step 1: Extract using LLM extractor
attributes = extract_attributes_with_llm(product_description, pipe_columns, flange_columns)
# print("Extracted Attributes:\n", json.dumps(attributes, indent=2))

# Step 2: Validate using LLM
validation_status = validate_attributes_with_llm(validated_result)
# print("Validation:", validation_status)

# Step 3: Store in correct variable based on PRODUCT
product_type = validated_result.get("PRODUCT", "").strip().upper()

Pipe_variable = None
Flange_variable = None

if "PIPE" in product_type:
    Pipe_variable = validated_result
    print("Stored in Pipe_variable ✅")
elif "FLANGE" in product_type:
    Flange_variable = validated_result
    print("Stored in Flange_variable ✅")
else:
    print("Unknown Product Type - Not stored ❌")


Stored in Flange_variable ✅


In [66]:
print(Pipe_variable)

None


Step 6: Norm-Based Attribute Recovery (Pipe Category Only)

In [47]:
Norm_template = pd.read_excel("ICE ENHANCEMENT PROJECT.xlsx", sheet_name="Norm-Std-pipe", header=None)
Norm_columns = Norm_template.iloc[1].tolist()

In [48]:
def validate_pipe_with_norm_lookup(Pipe_variable, norm_template_df):

    if not Pipe_variable:
        return "No Pipe data found."

    # Use only the top 5 rows of norm_template for context
    norm_template_preview = norm_template_df.to_string(index=False)

    prompt = f"""
You are a smart validation engine for PIPE product data.

Goal:
- If the PRODUCT is PIPE and any of the mandatory fields are "NA",
  use the provided Norm lookup table to fill in missing values.

Mandatory Fields for PIPE:
- PRODUCT
- NORM
- GRADE
- LEVEL_CLASS
- SIZE1
- SCHEDULE
- CONSTRUCTION

If any of these are "NA":
1. Check if a valid NORM (like A53, A106, API 5L) exists in the attributes.
2. If present, search the Norm lookup table for matching values.
3. Use the table to infer missing values like:
   - MATERIAL
   - CONSTRUCTION   
   - DIMEN_STAND

After filling:
- Re-check if all mandatory fields are complete (not "NA").

Respond with:
- "Valid Description (with Norm reference)" if everything is valid.
- "Invalid Description - Missing: <list of missing fields>" if still invalid.

Only use the Norm table if NORM is present in the attributes.

---
📦 Input PIPE Attributes:
{json.dumps(Pipe_variable, indent=2)}

📋 Norm Lookup Table Preview (first 5 rows):
{norm_template_preview}

Respond only with the validation message.
"""

    try:
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            temperature=0.1,
            messages=[
                {"role": "system", "content": "You validate PIPE product attributes using Norm-based rules."},
                {"role": "user", "content": prompt}
            ]
        )
        return response["choices"][0]["message"]["content"].strip()
    except Exception as e:
        print(f"Norm-Based LLM Error: {e}")
        return "Validation Error - Norm lookup failed"


In [49]:
Norm_template = pd.read_excel("ICE ENHANCEMENT PROJECT.xlsx", sheet_name="Norm-Std-pipe", header=None)
Norm_template.columns = Norm_template.iloc[1]  # Set column names from second row
Norm_template = Norm_template[2:]              # Drop the first two rows

norm_validation_status = validate_pipe_with_norm_lookup(Pipe_variable, Norm_template)
print("Norm Validation:", norm_validation_status)


Norm Validation: Invalid Description - Missing: NORM, GRADE, LEVEL_CLASS
